In [1]:
import math
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
def masked_softmax(X, valid_lens):
    # X.shape=(批量个数，查询长度，键值对个数)，也有可能查询长度为1，
    # Valid_lens.shape=(1) or (查询长度, 1)
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape # (批量个数, 查询长度，键值对个数)
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens, shape[1]) # shape=(查询长度*批量个数, )
        else:
            valid_lens = valid_lens.reshape(-1)
        X = d2l.sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6) 
        # X拉成shape=(查询长度*批量个数，键值对个数)
        # 用一个非常大的负数，是的softmax在后面几位输出位0
        return nn.functional.softmax(X.reshape(shape), dim=-1) # 还原X

In [ ]:
def sequence_mask(X, valid_len, value=0):
    """Mask irrelevant entries in sequences.

    Defined in :numref:`sec_utils`"""
    maxlen = X.size(1) # X每个序列的最大长度
    mask = torch.arange((maxlen), 
                        dtype=torch.float32, device=X.device)[None, :] < valid_len[:, None] 
    # 做一轮广播，maxlen沿着0轴广播valid_len次，valid_len沿着1轴广播maxlen次
    # mask.shape=(valid_len, maxlen)，这里valid_len
    X[~mask] = value # 翻转mask，所有大于等于valid_len的位置都赋值为value
    return X

In [2]:
class AdditiveAttention(nn.Module):
    def __init__(self, key_size, query_size, num_hiddens, dropout, **kwargs):
        super().__init__(**kwargs)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)
        self.w_v = nn.Linear(num_hiddens, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        queries, keys = self.W_q(queries), self.W_k(keys)
        # queries.shape=(batch_size, query_size, num_hiddens)
        # keys.shape=(batch_size, keys_size, num_hiddens)
        # 这两个没法直接相加，所以要进行广播处理
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        # features.shape=(batch_size, queries_size, keys_size, num_hiddens)
        features = torch.tanh(features)
        scores = self.w_v(features).squeeze(-1) # 注意力分数
        # scores.shape=(batch_size, queries_size, keys_size)
        self.attention_weights = masked_softmax(scores, valid_lens) # 在keys_size维度进行softmax
        # attention_weights.shape=(batch_size, queries_size, keys_size)
        # values.shape=(batch_size, key_size, value_len)
        return torch.bmm(self.dropout(self.attention_weights), values)
        # return.shape=(batch_size, queries_size, value_len) 所有的值代表室注意力机制处理过之后的值

In [3]:
class DotProductAttention(nn.Module):
    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        # queries.shape=(批量数，查询个数，长度d)
        # keys.shape=(批量数，键值对个数，长度d)
        # values.shape=(批量数，键值对个数，值长度)
        # valid_len=(批量数，) or (批量数，查询个数)
        d = queries.shape[-1]
        socres = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        # socres.shape=(批量数，查询个数，键值对个数)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)
        # return.shape=(批量数，查询个数，值长度)